# C12-classical-models — Practice p11 — Solution


Each node independently enumerates the deterministic Gini candidates. Stopping is checked before splitting, and class-count ties choose the smaller label.


In [ ]:
import numpy as np

X_p11 = np.array([[0.,0.],[0.,1.],[1.,0.],[1.,1.],
                  [2.,0.],[2.,1.],[3.,0.],[3.,1.]], dtype=np.float64)
y_p11 = np.array([0,0,0,1,1,1,1,0], dtype=np.int64)


def fit_depth_limited_tree(X, y, max_depth=2, min_samples_split=2):
    if not isinstance(X, np.ndarray) or X.dtype != np.float64 or X.ndim != 2:
        raise ValueError("X must be a float64 matrix")
    if not isinstance(y, np.ndarray) or not np.issubdtype(y.dtype, np.integer) or y.ndim != 1:
        raise ValueError("y must be an integer vector")
    if X.shape[0] < 1 or X.shape[1] < 1 or y.shape != (X.shape[0],) or not np.isfinite(X).all():
        raise ValueError("invalid data")
    if not isinstance(max_depth, (int, np.integer)) or isinstance(max_depth, bool) or max_depth < 0:
        raise ValueError("max_depth must be nonnegative")
    if not isinstance(min_samples_split, (int, np.integer)) or isinstance(min_samples_split, bool) or min_samples_split <= 0:
        raise ValueError("min_samples_split must be positive")
    def leaf(labels):
        classes, counts = np.unique(labels, return_counts=True)
        return {"prediction": int(classes[np.flatnonzero(counts == counts.max())[0]])}
    def impurity(labels):
        counts = np.unique(labels, return_counts=True)[1].astype(float)
        p = counts / labels.size
        return float(1.0 - p @ p)
    def build(rows, depth):
        labels = y[rows]
        if np.unique(labels).size == 1 or depth == max_depth or rows.size < min_samples_split:
            return leaf(labels)
        parent = impurity(labels)
        candidates = []
        for feature in range(X.shape[1]):
            values = np.unique(X[rows, feature])
            for threshold in (values[:-1] + values[1:]) / 2.0:
                local_left = X[rows, feature] <= threshold
                count = int(local_left.sum())
                if count in (0, rows.size):
                    continue
                weighted = (count * impurity(labels[local_left]) + (rows.size-count) * impurity(labels[~local_left])) / rows.size
                gain = parent - weighted
                if gain > 0.0:
                    candidates.append((float(weighted), feature, float(threshold), local_left))
        if not candidates:
            return leaf(labels)
        _, feature, threshold, local_left = min(candidates, key=lambda value: value[:3])
        return {"feature": int(feature), "threshold": float(threshold),
                "left": build(rows[local_left], depth + 1),
                "right": build(rows[~local_left], depth + 1)}
    return build(np.arange(X.shape[0]), 0)


def predict_tree(tree, X):
    if not isinstance(X, np.ndarray) or X.dtype != np.float64 or X.ndim != 2 or X.shape[1] < 1 or not np.isfinite(X).all():
        raise ValueError("X must be a finite float64 matrix")
    result = np.empty(X.shape[0], dtype=np.int64)
    for row_index, row in enumerate(X):
        node = tree
        while "prediction" not in node:
            node = node["left"] if row[node["feature"]] <= node["threshold"] else node["right"]
        result[row_index] = node["prediction"]
    return result


tree_p11 = fit_depth_limited_tree(X_p11, y_p11)
predictions_p11 = predict_tree(tree_p11, X_p11)


### Answer check


In [ ]:
expected_tree_p11 = {"feature": 0, "threshold": 0.5,
    "left": {"prediction": 0},
    "right": {"feature": 0, "threshold": 1.5,
              "left": {"prediction": 0}, "right": {"prediction": 1}}}
assert tree_p11 == expected_tree_p11
assert np.array_equal(predictions_p11, [0,0,0,0,1,1,1,1])
assert predictions_p11.dtype == np.int64
